<a href="https://colab.research.google.com/github/ibrahimbarghout/robust-ecg-domain-generalization/blob/main/notebooks/03_dataset_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# PTB-XL Research Project
## 3. Dataset Exploration

#This notebook explores the structure, labels, demographics,
#recording characteristics, rhythm annotations, and signal-quality
#annotations of the PTB-XL dataset.

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import pandas as pd
import numpy as np

from ast import literal_eval
from collections import Counter

In [5]:
PROJECT_PATH = "/content/drive/MyDrive/PTB-XL Research Project"

DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
)

print("Dataset path:")
print(DATA_PATH)

Dataset path:
/content/drive/MyDrive/PTB-XL Research Project/data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3


In [6]:
df = pd.read_csv(
    os.path.join(DATA_PATH, "ptbxl_database.csv"),
    index_col="ecg_id"
)

scp = pd.read_csv(
    os.path.join(DATA_PATH, "scp_statements.csv"),
    index_col=0
)

print("Number of ECG records:", len(df))
print("Number of columns:", len(df.columns))
print("Number of SCP statements:", len(scp))

Number of ECG records: 21799
Number of columns: 27
Number of SCP statements: 71


In [7]:
### 3.1 Dataset Structure

In [8]:
print("Columns:")
print(df.columns.tolist())

print("\nNumber of ECG records:", len(df))
print("Number of unique patients:", df["patient_id"].nunique())

Columns:
['patient_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site', 'device', 'recording_date', 'report', 'scp_codes', 'heart_axis', 'infarction_stadium1', 'infarction_stadium2', 'validated_by', 'second_opinion', 'initial_autogenerated_report', 'validated_by_human', 'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems', 'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr']

Number of ECG records: 21799
Number of unique patients: 18869


In [9]:
### 3.2 SCP Diagnostic and Rhythm Codes

In [10]:
scp_codes = df["scp_codes"].apply(literal_eval)

code_counts = Counter()

for codes in scp_codes:
    code_counts.update(codes.keys())

print("Number of unique SCP codes:", len(code_counts))

print("\nMost common SCP codes:")

for code, count in code_counts.most_common(30):
    print(f"{code}: {count}")

Number of unique SCP codes: 71

Most common SCP codes:
SR: 16748
NORM: 9514
ABQRS: 3327
IMI: 2676
ASMI: 2357
LVH: 2132
NDT: 1825
LAFB: 1623
AFIB: 1514
ISC_: 1272
PVC: 1143
IRBBB: 1118
STD_: 1009
VCLVH: 875
STACH: 826
1AVB: 793
IVCD: 787
SARRH: 772
NST_: 767
ISCAL: 659
SBRAD: 637
QWAVE: 548
CRBBB: 541
CLBBB: 536
ILMI: 478
LOWT: 438
LAO/LAE: 426
NT_: 423
PAC: 398
AMI: 353


In [11]:
### 3.3 SCP Statement Definitions

In [12]:
print("Number of SCP statements:", len(scp))

print("\nColumns:")
print(scp.columns.tolist())

display(
    scp[
        [
            "description",
            "diagnostic",
            "form",
            "rhythm",
            "diagnostic_class",
            "diagnostic_subclass"
        ]
    ].head(20)
)

Number of SCP statements: 71

Columns:
['description', 'diagnostic', 'form', 'rhythm', 'diagnostic_class', 'diagnostic_subclass', 'Statement Category', 'SCP-ECG Statement Description', 'AHA code', 'aECG REFID', 'CDISC Code', 'DICOM Code']


,description,diagnostic,form,rhythm,diagnostic_class,diagnostic_subclass
NDT,non-diagnostic T abnormalities,1.0,1.0,NaN,STTC,STTC
NST_,non-specific ST changes,1.0,1.0,NaN,STTC,NST_
DIG,digitalis-effect,1.0,1.0,NaN,STTC,STTC
LNGQT,long QT-interval,1.0,1.0,NaN,STTC,STTC
NORM,normal ECG,1.0,NaN,NaN,NORM,NORM
IMI,inferior myocardial infarction,1.0,NaN,NaN,MI,IMI
ASMI,anteroseptal myocardial infarction,1.0,NaN,NaN,MI,AMI
LVH,left ventricular hypertrophy,1.0,NaN,NaN,HYP,LVH
LAFB,left anterior fascicular block,1.0,NaN,NaN,CD,LAFB/LPFB
ISC_,non-specific ischemic,1.0,NaN,NaN,STTC,ISC_


In [13]:
### 3.4 Diagnostic Classes

In [14]:
diagnostic_classes = scp[
    scp["diagnostic"] == 1
][
    ["description", "diagnostic_class", "diagnostic_subclass"]
]

display(diagnostic_classes)

,description,diagnostic_class,diagnostic_subclass
NDT,non-diagnostic T abnormalities,STTC,STTC
NST_,non-specific ST changes,STTC,NST_
DIG,digitalis-effect,STTC,STTC
LNGQT,long QT-interval,STTC,STTC
NORM,normal ECG,NORM,NORM
IMI,inferior myocardial infarction,MI,IMI
ASMI,anteroseptal myocardial infarction,MI,AMI
LVH,left ventricular hypertrophy,HYP,LVH
LAFB,left anterior fascicular block,CD,LAFB/LPFB
ISC_,non-specific ischemic,STTC,ISC_


In [15]:
print("Diagnostic classes:")
print(
    diagnostic_classes["diagnostic_class"]
    .value_counts(dropna=True)
)

Diagnostic classes:
diagnostic_class
MI      14
STTC    13
CD      11
HYP      5
NORM     1
Name: count, dtype: int64


In [16]:
### 3.5 Patient Distribution

In [17]:
ecgs_per_patient = df.groupby("patient_id").size()

print("Total ECG recordings:", len(df))
print("Unique patients:", df["patient_id"].nunique())

print("\nECGs per patient:")
print(ecgs_per_patient.describe())

print(
    "\nPatients with multiple ECGs:",
    (ecgs_per_patient > 1).sum()
)

print(
    "Maximum ECGs from one patient:",
    ecgs_per_patient.max()
)

Total ECG recordings: 21799
Unique patients: 18869

ECGs per patient:
count    18869.000000
mean         1.155281
std          0.523105
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         10.000000
dtype: float64

Patients with multiple ECGs: 2111
Maximum ECGs from one patient: 10


In [18]:
### 3.6 Stratified Fold Structure

In [19]:
print("ECGs per fold:")
print(df["strat_fold"].value_counts().sort_index())

print("\nUnique patients per fold:")
print(
    df.groupby("strat_fold")["patient_id"]
      .nunique()
      .sort_index()
)

ECGs per fold:
strat_fold
1     2175
2     2181
3     2192
4     2174
5     2174
6     2173
7     2176
8     2173
9     2183
10    2198
Name: count, dtype: int64

Unique patients per fold:
strat_fold
1     1878
2     1849
3     1887
4     1883
5     1890
6     1884
7     1871
8     1881
9     1942
10    1904
Name: patient_id, dtype: int64


In [20]:
patient_fold_count = (
    df.groupby("patient_id")["strat_fold"]
      .nunique()
)

print(
    "Patients appearing in more than one fold:",
    (patient_fold_count > 1).sum()
)

print(
    "Maximum number of folds containing the same patient:",
    patient_fold_count.max()
)

Patients appearing in more than one fold: 0
Maximum number of folds containing the same patient: 1


In [21]:
### 3.7 ECG Quality and Artifact Annotations

In [22]:
quality_columns = [
    "baseline_drift",
    "static_noise",
    "burst_noise",
    "electrodes_problems",
    "extra_beats",
    "pacemaker"
]

for column in quality_columns:
    count = df[column].notna().sum()
    percentage = count / len(df) * 100

    print(
        f"{column:20s}: "
        f"{count:5d} ECGs "
        f"({percentage:5.1f}%)"
    )

baseline_drift      :  1598 ECGs (  7.3%)
static_noise        :  3260 ECGs ( 15.0%)
burst_noise         :   613 ECGs (  2.8%)
electrodes_problems :    30 ECGs (  0.1%)
extra_beats         :  1949 ECGs (  8.9%)
pacemaker           :   291 ECGs (  1.3%)


In [23]:
### 3.8 Rhythm Annotations

In [24]:
rhythm_codes = scp[
    scp["rhythm"] == 1
]["description"]

rhythm_counts = {}

for code, count in code_counts.items():
    if code in rhythm_codes.index:
        rhythm_counts[code] = count

for code, count in sorted(
    rhythm_counts.items(),
    key=lambda x: x[1],
    reverse=True
):
    description = scp.loc[code, "description"]
    print(
        f"{code:6s} "
        f"{count:6,d} "
        f"{description}"
    )

SR     16,748 sinus rhythm
AFIB    1,514 atrial fibrillation
STACH     826 sinus tachycardia
SARRH     772 sinus arrhythmia
SBRAD     637 sinus bradycardia
PACE      294 normal functioning artificial pacemaker
SVARR     157 supraventricular arrhythmia
BIGU       82 bigeminal pattern (unknown origin, SV or Ventricular)
AFLT       73 atrial flutter
SVTAC      27 supraventricular tachycardia
PSVT       24 paroxysmal supraventricular tachycardia
TRIGU      20 trigeminal pattern (unknown origin, SV or Ventricular)
